# Notebook 5: Data Mesh

**Pattern:** Organizational pattern — domain teams own their data as products. Decentralized ownership, federated governance.

**Stack:** DuckDB + simulated per-domain catalogs with data product contracts (JSON schemas)

**Maple Trust Bank** — synthetic BFSI data

---
## Configuration

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
DATA_DIR = "../data"
LINEAGE_PATH = f"{DATA_DIR}/lineage/lineage_graph.json"
MESH_DIR = "/tmp/data_mesh_demo"

print("Data Mesh simulation — three banking domains with data product contracts.")
print(f"Mesh workspace: {MESH_DIR}")

In [ ]:
import duckdb
import pandas as pd
import json
import os
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

---
## Section 1: The Pattern in One Paragraph

**Data mesh** is not a technology — it's an organizational pattern. Four principles, coined by Zhamak Dehghani: (1) domain ownership — the team that creates data owns it, (2) data as a product — domains publish curated, documented, SLA-backed data products, (3) self-serve data platform — infrastructure that makes it easy for domains to publish, (4) federated computational governance — global standards enforced locally. The idea is powerful: stop pretending a central team can understand and govern all data. Let the people who know the data best own it. The problem: this requires product thinking, organizational maturity, and investment that most companies don't have. Data mesh is the most over-hyped and under-implemented pattern in the industry. It's the right answer for organizations with 500+ data engineers and strong product culture. For everyone else, it's a conference talk.

---
## Section 2: When You'd Use It, When You Wouldn't

| Use when | Don't use when |
|----------|----------------|
| Large org with strong, autonomous domain teams | Small team (< 50 data people) |
| Clear domain boundaries aligned to business units | No domain ownership culture |
| Product thinking maturity (PMs for data) | Orgs without product thinking discipline |
| Central platform team can build self-serve infra | "We just want to query everything from one place" |
| Each domain has enough engineers to own a pipeline | Data engineering is fully centralized |
| Governance is mature enough to federate | Governance barely exists centrally |

---
## Section 3: The Setup — Simulated Data Mesh

We simulate a mesh with three banking domains:
- **Retail Banking** — owns customers, retail accounts, retail transactions
- **Commercial Banking** — owns commercial accounts, commercial transactions
- **Risk & Compliance** — owns AML alerts, risk scores, entity links

Each domain publishes a **data product** with a JSON schema contract.

> ⚠️ **This is a simulation.** Real data mesh requires organizational change, domain team autonomy, self-serve infrastructure, and federated governance. A notebook can show the consumer experience; it cannot show the org design.

In [ ]:
# Clean up previous runs
if os.path.exists(MESH_DIR):
    shutil.rmtree(MESH_DIR)

# Create domain directories (each domain owns its own namespace)
domains = ["retail_banking", "commercial_banking", "risk_compliance"]
for domain in domains:
    os.makedirs(os.path.join(MESH_DIR, domain), exist_ok=True)

print("Created domain directories:")
for d in domains:
    print(f"  {MESH_DIR}/{d}/")

In [ ]:
# Load raw data
branches_df = pd.read_parquet(f"{DATA_DIR}/branches.parquet")
customers_df = pd.read_parquet(f"{DATA_DIR}/customers.parquet")
accounts_df = pd.read_parquet(f"{DATA_DIR}/accounts.parquet")
transactions_df = pd.read_parquet(f"{DATA_DIR}/transactions.parquet")

print(f"Loaded raw data: branches={len(branches_df)}, customers={len(customers_df)}, "
      f"accounts={len(accounts_df)}, transactions={len(transactions_df)}")

In [ ]:
# Define data product contracts (JSON schema)
contracts = {
    "retail_banking": {
        "domain": "Retail Banking",
        "owner": "retail_data_team@mapletrust.ca",
        "sla": "99.5% availability, refreshed daily by 06:00 ET",
        "products": {
            "customers": {
                "description": "Retail customer master data",
                "schema": {
                    "required_fields": ["customer_id", "name", "segment", "kyc_status", "risk_score"],
                    "types": {"customer_id": "string", "name": "string", "segment": "string",
                              "kyc_status": "string", "risk_score": "integer"},
                    "constraints": {"segment": ["retail", "wealth"], "risk_score": "1-100"}
                },
                "row_count_range": [50000, 150000],
                "freshness": "daily"
            },
            "retail_accounts": {
                "description": "Retail banking accounts (chequing, savings)",
                "schema": {
                    "required_fields": ["account_id", "customer_id", "account_type", "status", "balance"],
                    "types": {"account_id": "string", "customer_id": "string", "account_type": "string",
                              "status": "string", "balance": "float"},
                    "constraints": {"account_type": ["chequing", "savings", "credit"]}
                },
                "freshness": "daily"
            },
            "retail_transactions": {
                "description": "Retail transaction events",
                "schema": {
                    "required_fields": ["transaction_id", "account_id", "branch_id", "amount", "timestamp"],
                    "types": {"transaction_id": "string", "account_id": "string", "branch_id": "string",
                              "amount": "float", "timestamp": "datetime"}
                },
                "freshness": "near-real-time"
            }
        }
    },
    "commercial_banking": {
        "domain": "Commercial Banking",
        "owner": "commercial_data_team@mapletrust.ca",
        "sla": "99.0% availability, refreshed daily by 08:00 ET",
        "products": {
            "commercial_accounts": {
                "description": "Commercial banking accounts (investment, mortgage)",
                "schema": {
                    "required_fields": ["account_id", "customer_id", "account_type", "status", "balance"],
                    "types": {"account_id": "string", "customer_id": "string", "account_type": "string",
                              "status": "string", "balance": "float"},
                    "constraints": {"account_type": ["investment", "mortgage"]}
                },
                "freshness": "daily"
            },
            "commercial_transactions": {
                "description": "Commercial transaction events",
                "schema": {
                    "required_fields": ["transaction_id", "account_id", "branch_id", "amount", "timestamp"],
                    "types": {"transaction_id": "string", "account_id": "string", "branch_id": "string",
                              "amount": "float", "timestamp": "datetime"}
                },
                "freshness": "daily"
            }
        }
    },
    "risk_compliance": {
        "domain": "Risk & Compliance",
        "owner": "risk_data_team@mapletrust.ca",
        "sla": "99.9% availability, refreshed daily by 05:00 ET",
        "products": {
            "aml_alerts": {
                "description": "AML alerts generated by transaction monitoring",
                "schema": {
                    "required_fields": ["alert_id", "customer_id", "alert_type", "risk_level", "created_at"],
                    "types": {"alert_id": "string", "customer_id": "string", "alert_type": "string",
                              "risk_level": "string", "created_at": "datetime"}
                },
                "freshness": "near-real-time"
            },
            "customer_risk_scores": {
                "description": "ML-derived customer risk scores for AML/KYC",
                "schema": {
                    "required_fields": ["customer_id", "risk_score", "risk_category", "scored_at"],
                    "types": {"customer_id": "string", "risk_score": "integer",
                              "risk_category": "string", "scored_at": "datetime"},
                    "constraints": {"risk_category": ["low", "medium", "high", "very_high"]}
                },
                "freshness": "daily"
            }
        }
    }
}

# Write contracts to domain directories
for domain_key, contract in contracts.items():
    contract_path = os.path.join(MESH_DIR, domain_key, "contract.json")
    with open(contract_path, "w") as f:
        json.dump(contract, f, indent=2)
    print(f"Published contract: {domain_key} ({len(contract['products'])} data products)")

print("\nEach domain owns its data products and publishes a contract.")
print("Consumers discover products via the contract, not by exploring raw files.")

In [ ]:
# Partition and publish data per domain
import numpy as np

# Retail Banking: retail + wealth customers, chequing/savings/credit accounts
retail_customers = customers_df[customers_df["segment"].isin(["retail", "wealth"])].copy()
retail_account_types = ["chequing", "savings", "credit"]
retail_accounts = accounts_df[accounts_df["account_type"].isin(retail_account_types)].copy()
retail_acct_ids = set(retail_accounts["account_id"])
retail_transactions = transactions_df[transactions_df["account_id"].isin(retail_acct_ids)].copy()

retail_customers.to_parquet(os.path.join(MESH_DIR, "retail_banking", "customers.parquet"))
retail_accounts.to_parquet(os.path.join(MESH_DIR, "retail_banking", "retail_accounts.parquet"))
retail_transactions.to_parquet(os.path.join(MESH_DIR, "retail_banking", "retail_transactions.parquet"))

print(f"Retail Banking domain:")
print(f"  customers:           {len(retail_customers):,}")
print(f"  retail_accounts:     {len(retail_accounts):,}")
print(f"  retail_transactions: {len(retail_transactions):,}")

In [ ]:
# Commercial Banking: commercial + institutional customers, investment/mortgage accounts
commercial_account_types = ["investment", "mortgage"]
commercial_accounts = accounts_df[accounts_df["account_type"].isin(commercial_account_types)].copy()
commercial_acct_ids = set(commercial_accounts["account_id"])
commercial_transactions = transactions_df[transactions_df["account_id"].isin(commercial_acct_ids)].copy()

commercial_accounts.to_parquet(os.path.join(MESH_DIR, "commercial_banking", "commercial_accounts.parquet"))
commercial_transactions.to_parquet(os.path.join(MESH_DIR, "commercial_banking", "commercial_transactions.parquet"))

print(f"Commercial Banking domain:")
print(f"  commercial_accounts:     {len(commercial_accounts):,}")
print(f"  commercial_transactions: {len(commercial_transactions):,}")

In [ ]:
# Risk & Compliance: generate synthetic AML alerts and risk scores
np.random.seed(42)

# Generate AML alerts for high-risk customers
high_risk_customers = customers_df[customers_df["risk_score"] >= 60]["customer_id"].values
n_alerts = min(5000, len(high_risk_customers) * 2)
alert_customer_ids = np.random.choice(high_risk_customers, size=n_alerts, replace=True)

aml_alerts = pd.DataFrame({
    "alert_id": [f"AML-{i+1:06d}" for i in range(n_alerts)],
    "customer_id": alert_customer_ids,
    "alert_type": np.random.choice(
        ["unusual_volume", "structuring", "high_risk_jurisdiction", "sanctions_match", "pattern_anomaly"],
        size=n_alerts
    ),
    "risk_level": np.random.choice(["low", "medium", "high", "critical"], size=n_alerts, p=[0.3, 0.35, 0.25, 0.1]),
    "created_at": pd.date_range("2024-01-01", periods=n_alerts, freq="h"),
})

# Generate customer risk scores
risk_scores = pd.DataFrame({
    "customer_id": customers_df["customer_id"].values,
    "risk_score": customers_df["risk_score"].values,
    "risk_category": pd.cut(customers_df["risk_score"],
                            bins=[0, 25, 50, 75, 100],
                            labels=["low", "medium", "high", "very_high"]).astype(str),
    "scored_at": pd.Timestamp("2024-10-01"),
})

aml_alerts.to_parquet(os.path.join(MESH_DIR, "risk_compliance", "aml_alerts.parquet"))
risk_scores.to_parquet(os.path.join(MESH_DIR, "risk_compliance", "customer_risk_scores.parquet"))

print(f"Risk & Compliance domain:")
print(f"  aml_alerts:           {len(aml_alerts):,}")
print(f"  customer_risk_scores: {len(risk_scores):,}")

In [ ]:
# Show the mesh directory structure
print("📊 Reference Architecture Swimlane: Discovery & Exploration")
print("   (IBM Knowledge Catalog — federated domain catalogs)\n")

print("Data Mesh directory structure:")
for domain in domains:
    domain_path = os.path.join(MESH_DIR, domain)
    files = os.listdir(domain_path)
    print(f"\n  {domain}/")
    for f in sorted(files):
        fpath = os.path.join(domain_path, f)
        size = os.path.getsize(fpath) / 1024 / 1024
        print(f"    {f:40s} {size:>8.1f} MB")

---
## Section 4: Three Canonical Queries

In [ ]:
# Connect DuckDB
con = duckdb.connect()

### Q1: Total transaction volume by branch for Q3 2024

Consumer queries the **Retail Banking** domain's data product. Clean, curated, SLA-backed.

In [ ]:
# First: read the contract to understand what we're consuming
with open(os.path.join(MESH_DIR, "retail_banking", "contract.json")) as f:
    retail_contract = json.load(f)

print("Data Product Contract — Retail Banking")
print(f"  Owner: {retail_contract['owner']}")
print(f"  SLA:   {retail_contract['sla']}")
print(f"  Products: {list(retail_contract['products'].keys())}")
print()
print("As a consumer, I discover and trust the data through its contract.")
print("I don't need to know HOW it was built — only WHAT it provides.")

In [ ]:
# Query the retail banking domain's transaction product
q1 = con.execute(f"""
    SELECT
        t.branch_id,
        COUNT(*)           AS txn_count,
        SUM(t.amount)      AS total_amount,
        AVG(t.amount)      AS avg_amount
    FROM '{MESH_DIR}/retail_banking/retail_transactions.parquet' t
    WHERE t.timestamp >= '2024-07-01' AND t.timestamp < '2024-10-01'
    GROUP BY t.branch_id
    ORDER BY total_amount DESC
""").fetchdf()

print("Q1: Retail transaction volume by branch — Q3 2024 (from Retail Banking domain)")
print("Domain owns and curates this data. Consumer trusts the contract.")
q1.head(10)

### Q2: Cross-domain query — Join Retail (customers) + Risk (AML alerts)

Consumer needs data from TWO domains. This requires understanding both contracts.

In [ ]:
# Read both contracts
with open(os.path.join(MESH_DIR, "risk_compliance", "contract.json")) as f:
    risk_contract = json.load(f)

print("Cross-domain query requires understanding TWO contracts:")
print()
print(f"  Domain 1: {retail_contract['domain']}")
print(f"    Product: customers")
print(f"    Join key: customer_id (string)")
print()
print(f"  Domain 2: {risk_contract['domain']}")
print(f"    Product: aml_alerts")
print(f"    Join key: customer_id (string)")
print()
print("The shared key 'customer_id' is a governance challenge:")
print("  → Who owns the definition of customer_id?")
print("  → What if the domains use different ID formats?")
print("  → This is why federated governance is the hardest mesh principle.")

In [ ]:
q2 = con.execute(f"""
    SELECT
        c.customer_id,
        c.name,
        c.segment,
        c.risk_score AS retail_risk_score,
        a.alert_id,
        a.alert_type,
        a.risk_level,
        a.created_at
    FROM '{MESH_DIR}/retail_banking/customers.parquet' c
    JOIN '{MESH_DIR}/risk_compliance/aml_alerts.parquet' a
        ON c.customer_id = a.customer_id
    WHERE a.risk_level IN ('high', 'critical')
    ORDER BY a.created_at DESC
""").fetchdf()

print(f"Q2: Cross-domain join — Retail customers with high/critical AML alerts")
print(f"    Results: {len(q2):,} rows (customers with high/critical alerts)")
q2.head(10)

### Q3: Lineage — within-domain vs. cross-domain

Each domain owns its own lineage. Cross-domain lineage is the hard problem.

In [ ]:
# Within-domain lineage: Risk & Compliance knows exactly how AML alerts are built
print("Q3: Lineage")
print("="*60)
print()
print("WITHIN-DOMAIN lineage (Risk & Compliance):")
print("  raw.transactions → curated.transactions_clean → consumed.aml_alerts")
print("  raw.customers → curated.customer_360 → consumed.aml_alerts")
print("  raw.wire_transfers → curated.wire_transfers_enriched → consumed.aml_alerts")
print("  curated.sanctions_reference → consumed.aml_alerts")
print()
print("  ✅ The Risk domain knows exactly how its alerts are produced.")
print("     It owns the pipeline, the data, and the lineage.")

In [ ]:
# Cross-domain lineage: the hard problem
with open(LINEAGE_PATH) as f:
    lineage = json.load(f)

print("\nCROSS-DOMAIN lineage (the hard problem):")
print()
print("  Our Q2 query joins:")
print("    Retail Banking → customers (owned by retail_data_team)")
print("    Risk & Compliance → aml_alerts (owned by risk_data_team)")
print()
print("  Questions no single domain can answer:")
print("    → What is the full lineage of Q2's result?")
print("    → If retail changes their customer_id format, who breaks?")
print("    → If risk changes their alert thresholds, does retail know?")
print()
print("  This requires a FEDERATED CATALOG that spans domains.")
print("  → IBM Knowledge Catalog provides this in the IBM stack.")
print()

# Show the portion of the lineage graph relevant to aml_alerts
target = "consumed.aml_alerts"
upstream_edges = [e for e in lineage["edges"] if e["to"] == target]
print(f"Lineage graph edges into {target}:")
for edge in upstream_edges:
    node = next((n for n in lineage["nodes"] if n["id"] == edge["from"]), None)
    owner = node["owner"] if node else "unknown"
    print(f"  ← {edge['from']:40s} (owner: {owner})")

---
## Section 5: Where This Pattern Breaks

### Break 1: Domains don't invest in data quality

A data product that violates its own contract.

In [ ]:
# Create a "bad" data product that violates its contract
bad_product = pd.DataFrame({
    "customer_id": ["CUST-000001", None, "CUST-000003", "CUST-INVALID", "CUST-000005"],
    "name": ["Alice", "Bob", None, "Diana", "Eve"],                  # missing name
    # 'segment' is REQUIRED by contract — but missing entirely
    # 'kyc_status' is REQUIRED by contract — but missing entirely
    "risk_score": [25, -5, 50, 200, None],                            # invalid values
})

print("A 'bad' data product from a domain that didn't invest in quality:")
print(bad_product)
print()
print("Contract violations:")
print("  ✗ customer_id has NULL (row 2)")
print("  ✗ name has NULL (row 3)")
print("  ✗ 'segment' field is MISSING (required by contract)")
print("  ✗ 'kyc_status' field is MISSING (required by contract)")
print("  ✗ risk_score = -5 (must be 1-100)")
print("  ✗ risk_score = 200 (must be 1-100)")
print("  ✗ risk_score = NULL (must be integer)")

In [ ]:
# Validate against contract
contract = contracts["retail_banking"]["products"]["customers"]
required_fields = contract["schema"]["required_fields"]
actual_fields = list(bad_product.columns)

print("Contract validation:")
print(f"  Required fields: {required_fields}")
print(f"  Actual fields:   {actual_fields}")
print()

missing = set(required_fields) - set(actual_fields)
if missing:
    print(f"  FAILED: Missing required fields: {missing}")

# Check nulls in required fields
for col in actual_fields:
    if col in required_fields and bad_product[col].isna().any():
        null_count = bad_product[col].isna().sum()
        print(f"  FAILED: '{col}' has {null_count} NULL(s) — contract requires non-null")

print()
print("⚠️  Without enforcement, contracts are just documentation.")
print("   Data mesh assumes domains WILL invest in quality.")
print("   When they don't, consumers get garbage with a nice label.")

### Break 2: Cross-domain queries with no shared keys

In [ ]:
# What if commercial banking uses a different customer ID format?
print("Break 2: Cross-domain join with incompatible keys")
print("="*60)
print()
print("Scenario: Retail uses customer_id = 'CUST-000001'")
print("          Commercial uses client_ref = 'CL-MTB-000001'")
print("          Risk uses entity_id = 'ENT-000001'")
print()
print("Each domain chose its own identifier scheme.")
print("There is no global entity resolution.")
print("Cross-domain joins produce ZERO results or require MDM.")
print()
print("This is the #1 failure mode of data mesh:")
print("  Domains are autonomous — but autonomy without standards = chaos.")
print("  Federated governance MUST include shared key standards.")
print("  → IBM Master Data Management (MDM) solves entity resolution.")

### Break 3: Federated governance — theory vs. practice

In [ ]:
print("Break 3: Federated governance in practice")
print("="*60)
print()
print("Theory:")
print("  → Each domain enforces global policies locally")
print("  → A governance council sets standards")
print("  → Automated compliance checks run at publish time")
print()
print("Practice:")
print("  → The governance council meets quarterly (if that)")
print("  → Standards exist in a wiki no one reads")
print("  → Domains skip quality checks when under deadline pressure")
print("  → PII leaks across domain boundaries via join keys")
print("  → No one owns cross-domain data quality")
print()
print('"Data mesh is the right answer for orgs with 500+ data engineers')
print('and strong product culture. For everyone else, it\'s a conference')
print('talk that became a reorg."')

---
## Section 6: The IBM Stack Mapping

| Component | IBM Product | Swimlane |
|-----------|-------------|----------|
| Federated Catalog | IBM Knowledge Catalog (per-domain catalogs) | Discovery & Exploration |
| Self-Serve Platform | watsonx.data (domain-level compute) | Analytical Data Management & Storage |
| Data Quality | IBM Data Quality (automated contract validation) | Information & Model Management & Governance |
| Business Glossary | IBM Business Glossary (shared terms across domains) | Information & Model Management & Governance |
| Data Product APIs | IBM API Connect (publish data products as APIs) | Data Access |
| Entity Resolution | IBM Match 360 / MDM (shared key management) | Information & Model Management & Governance |

**Swimlane:** Discovery & Exploration — IBM Knowledge Catalog is the federated catalog layer.

**Note:** "IBM doesn't sell 'data mesh.' IBM sells the platform that makes mesh possible — if your org is ready."

In [ ]:
print("📊 Reference Architecture Swimlane: Discovery & Exploration")
print("   (IBM Knowledge Catalog — federated domain catalogs)\n")
print("IBM Product Mapping:")
print("  Federated Catalog  → IBM Knowledge Catalog (per-domain catalogs)")
print("  Self-Serve Platform → watsonx.data (domain-level compute)")
print("  Data Quality       → IBM Data Quality (contract validation)")
print("  Business Glossary  → IBM Business Glossary (shared terms)")
print("  Data Product APIs  → IBM API Connect")
print("  Entity Resolution  → IBM Match 360 / MDM")
print()
print("IBM doesn't sell 'data mesh.'")
print("IBM sells the platform that makes mesh possible — if your org is ready.")

---
## Section 7: BFSI Reality Check

No Canadian bank has fully implemented data mesh. Several have adopted elements: RBC has domain-oriented data teams aligned to lines of business, Scotiabank has invested in data product catalogs, and TD has experimented with self-serve data platforms. The pattern works at the edges — a single domain (say, credit risk) that owns its data end-to-end and publishes curated products for downstream consumers. Where it fails: cross-domain use cases like enterprise-wide customer 360, where you need retail, commercial, wealth, and insurance data joined together. The shared key problem ("what is a customer?") becomes existential. The banks that succeed with mesh are the ones that start with one domain, prove the model, and grow organically. The ones that fail try to mesh everything at once — reorging the data team into domains without changing the culture, tooling, or incentives. That's not mesh; that's chaos with a Zhamak citation.

In [ ]:
# Clean up
con.close()
print("Notebook 5 complete.")
print()
print("Key takeaway: Data mesh is an organizational pattern, not a technology.")
print("It works when domains own their data as products with enforceable contracts.")
print("It fails when governance is federated in theory but absent in practice.")
print("Next: Notebook 6 (RAG) — finally answering Q2 with AI.")